In [ ]:
# %%
from huggingface_hub import hf_hub_download, HfApi, login
import re
import json
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ------------------------------
# 1. Login + Check Access
# ------------------------------
token = os.environ["HF_TOKEN"]
repo = "google/gemma-2-2b-it"

api = HfApi()
try:
    api.repo_info(repo_id=repo, token=token)
    print("✔ Token CAN access the model.")
except Exception as e:
    print("❌ Token CANNOT access the model.")
    print(e)

print("ABC")
login(token)

# ------------------------------
# 2. Load Model
# ------------------------------
model_name = "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=token)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    use_auth_token=token
)

# ------------------------------
# 3. Extraction Function (ROBUST JSON FIX)
# ------------------------------
def extract_drugs_adrs(text):

    prompt = f"""
You are a clinical information extraction system.

Extract the drug names and adverse drug events (ADEs) from the text below.

Return ONLY valid JSON in this exact format:

{{
  "drug_names": ["drug1", "drug2"],
  "adverse_effects": ["effect1", "effect2"]
}}

Text:
{text}

ONLY RETURN THE JSON. NO EXTRA TEXT.
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    output_tokens = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False
    )

    response = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

    # ------------------------------
    # Extract JSON reliably
    # ------------------------------

    # 1. Try Markdown block ```json ... ```
    md_match = re.search(r"```json\s*(\{.*?\})\s*```", response, re.DOTALL)
    if md_match:
        json_text = md_match.group(1)
    else:
        # 2. Fallback: extract first {...} block
        brace_match = re.search(r"\{.*\}", response, re.DOTALL)
        if brace_match:
            json_text = brace_match.group(0)
        else:
            print("⚠️ No JSON detected in:", response)
            return [], []

    # 3. Load JSON safely
    try:
        data = json.loads(json_text)
    except Exception as e:
        print("⚠️ JSON PARSE ERROR:", e)
        print("JSON TEXT WAS:", json_text)
        return [], []

    return data.get("drug_names", []), data.get("adverse_effects", [])


# ------------------------------
# 4. Load CSV
# ------------------------------
input_csv_path = "./data/LLaVA-Med/subset_of_all_ADR.csv"
output_csv_path = "./data/LLaVA-Med/Extracted_Drugs_ADRs_151125.csv"

df = pd.read_csv(input_csv_path)
texts = df["Preprocessed Posts"].fillna("").astype(str).tolist()

# ------------------------------
# 5. Run Extraction (PRINT RESULTS AFTER EVERY ROW)
# ------------------------------
extracted_drugs = []
extracted_adrs = []

for i, text in enumerate(texts):
    drugs, adrs = extract_drugs_adrs(text)

    drugs_str = ", ".join(drugs)
    adrs_str = ", ".join(adrs)

    extracted_drugs.append(drugs_str)
    extracted_adrs.append(adrs_str)

    # Print after every row
    print(f"\n===== Row {i+1} =====")
    print("TEXT:", text)
    print("DRUGS:", drugs_str)
    print("ADRs:", adrs_str)
    print("=====================")

df["Extracted Drug Names"] = extracted_drugs
df["Extracted ADRs"] = extracted_adrs

# ------------------------------
# 6. Save Output
# ------------------------------
df.to_csv(output_csv_path, index=False)
print("\n✔ DONE! File saved to:", output_csv_path)


In [ ]:
from huggingface_hub import login
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import json

# ------------------------------
# 1. Login + Load Model
# ------------------------------
token = os.environ["HF_TOKEN"]
model_name = "google/gemma-2-2b-it"

login(token)

tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=token)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    use_auth_token=token
)


# ------------------------------
# 2. Extraction Function
# ------------------------------
def extract_drugs_adrs(text):
    """
    Uses Gemma 2 (2B) to extract drug names and adverse drug reactions as JSON.
    """

    prompt = f"""
You are a clinical information extraction system.

Extract the drug names and adverse drug events (ADEs) from the text below.

Return ONLY valid JSON in this exact format:

{{
  "drug_names": ["drug1", "drug2"],
  "adverse_effects": ["effect1", "effect2"]
}}

Text:
{text}

ONLY RETURN THE JSON. NO EXTRA TEXT.
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    output_tokens = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False
    )

    response = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

    # Parse JSON
    try:
        print("RAW MODEL OUTPUT:\n", response)
        data = json.loads(response.strip())
    except:
        data = {"drug_names": [], "adverse_effects": []}

    return data["drug_names"], data["adverse_effects"]


# ------------------------------
# 3. Process ONE text
# ------------------------------
text = "avastin giving me nose bleeds and headaches"

drug_names, adverse_effects = extract_drugs_adrs(text)

print("\n=== Extraction Result ===")
print("Drugs:", drug_names)
print("ADRs:", adverse_effects)


In [ ]:
from huggingface_hub import login
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import json
import re

# ------------------------------
# 1. Login + Load Model
# ------------------------------
token = os.environ["HF_TOKEN"]
model_name = "google/gemma-2-2b-it"

login(token)

tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=token)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    use_auth_token=token
)


# ------------------------------
# 2. Extraction Function (FIXED)
# ------------------------------
def extract_drugs_adrs(text):
    prompt = f"""
You are a clinical information extraction system.

Extract the drug names and adverse drug events (ADEs) from the text below.

Return ONLY valid JSON in this exact format:

{{
  "drug_names": ["drug1", "drug2"],
  "adverse_effects": ["effect1", "effect2"]
}}

Text:
{text}

ONLY RETURN THE JSON. NO EXTRA TEXT.
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    output_tokens = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False
    )

    response = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

    print("\nRAW MODEL OUTPUT:\n", response)

    # ------------------------------
    #  Extract JSON reliably
    # ------------------------------

    # 1. Extract markdown JSON block ```json ... ```
    md_match = re.search(r"```json\s*(\{.*?\})\s*```", response, re.DOTALL)
    if md_match:
        json_text = md_match.group(1)
    else:
        # 2. Extract first { ... } block anywhere
        brace_match = re.search(r"\{.*\}", response, re.DOTALL)
        if brace_match:
            json_text = brace_match.group(0)
        else:
            return [], []   # completely failed

    # 3. Try to load JSON
    try:
        data = json.loads(json_text)
    except Exception as e:
        print("JSON PARSE ERROR:", e)
        return [], []

    drug_names = data.get("drug_names", [])
    adverse_effects = data.get("adverse_effects", [])

    return drug_names, adverse_effects


# ------------------------------
# 3. Process ONE text
# ------------------------------
text = "avastin giving me nose bleeds and headaches"

drug_names, adverse_effects = extract_drugs_adrs(text)

print("\n=== Extraction Result ===")
print("Drugs:", drug_names)
print("ADRs:", adverse_effects)


In [ ]:
from huggingface_hub import login
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ------------------------------
# 1. Login + Load Model
# ------------------------------
token = os.environ["HF_TOKEN"]
model_name = "google/gemma-2-2b-it"

login(token)

tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=token)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    use_auth_token=token
)

# ------------------------------
# 2. Simple generation test
# ------------------------------
prompt = "Write a short, friendly sentence about cats."

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

output_tokens = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.7
)

response = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

print("\n=== MODEL OUTPUT ===")
print(response)
